In [1]:
import pandas as pd

# Step 1
user_df = pd.read_csv("../data/ab_user_master_2000.csv")
exposure_df = pd.read_csv("../data/ab_exposure_log_2000.csv")
action_df = pd.read_csv("../data/ab_action_log_2000.csv")
user_level_df = pd.read_csv("../data/ab_user_level_final.csv")

In [2]:
# TODO:
# user_level_df를 variant, country, device 기준으로 그룹화하세요.
# - user_id는 사용자 수(count)
# - impression은 합계(sum)
# - click은 "1번 이상 클릭한 사용자 수"가 되도록 집계
# - purchase는 "1번 이상 구매한 사용자 수"가 되도록 집계
# - revenue는 합계(sum)
segment_df = user_level_df.groupby(["variant", "country", "device"]).agg({
    "user_id": 'count',
    "impression": 'sum',
    "click": 'sum',
    "purchase": 'sum',
    "revenue": 'sum'
}).reset_index()

In [3]:
# TODO:
# user_id 집계 컬럼 이름을 users로 변경하세요.
segment_df = segment_df.rename(columns={'user_id':'users'})

In [4]:
# TODO:
# 세그먼트 KPI를 계산하세요.
# CTR = click / impression
# Conversion = purchase / impression
# Revenue_per_user = revenue / users
segment_df["CTR"] = segment_df['click'] / segment_df['impression']
segment_df["Conversion"] = segment_df['purchase'] / segment_df['impression']
segment_df["Revenue_per_user"] = segment_df['revenue'] / segment_df['users']

In [5]:
# TODO:
# exposure_df와 action_df에 날짜 컬럼(date)을 생성하세요.
# event_time을 datetime으로 변환한 뒤 날짜만 추출합니다.
exposure_df["date"] = pd.to_datetime(exposure_df['event_time']).dt.date
action_df["date"] = pd.to_datetime(exposure_df['event_time']).dt.date

In [6]:
# TODO:
# action_df에 variant를 붙이세요.
# exposure_df에서 session_id와 variant만 선택한 뒤,
# session_id 기준 left merge를 수행합니다.
action_df = pd.merge(
    action_df,
    exposure_df[['session_id', 'variant']],
    on='session_id',
    how='left'
)

In [7]:
# TODO:
# 날짜(date), variant 기준으로 노출 수를 집계하세요.
daily_exposure = exposure_df.groupby(['date', 'variant']).count().reset_index()

In [8]:
# TODO:
# event_id 컬럼 이름을 impression으로 변경하세요.
daily_exposure = daily_exposure.rename(columns={'event_id':'impression'})

In [9]:
# TODO:
# action_df에서 click 이벤트와 purchase 이벤트를 각각 분리하세요.
click_df = action_df[action_df['event_type'] == 'click'].copy()
purchase_df = action_df[action_df['event_type'] == 'purchase'].copy()

In [10]:
# TODO:
# click 이벤트를 날짜와 variant 기준으로 집계하세요.
daily_click = click_df.groupby(['date','variant']).count().reset_index()

In [11]:
# TODO:
# purchase 이벤트를 날짜와 variant 기준으로 집계하세요.
# event_id는 purchase 수, amount는 revenue 합계입니다.
daily_purchase = purchase_df.groupby(['date','variant']).count().reset_index()

daily_purchase = daily_purchase.rename(columns={
    'session_id': 'purchase',
    'amount': 'revenue'
})

In [12]:
# TODO:
# 일별 exposure, click, purchase 집계를 merge하여 trend_df를 생성하세요.
# 기준 컬럼은 date, variant 입니다.
trend_df = pd.merge(
    daily_exposure,
    daily_click.rename(columns={'session_id': 'click'}),
    on=['date', 'variant'],
    how='left'
)
trend_df = pd.merge(
    trend_df,
    daily_purchase.rename(columns={'session_id': 'purchase'}),
    on=['date', 'variant'],
    how='left'
)

In [13]:
# TODO:
# date, variant 기준으로 정렬하세요.
trend_df = trend_df.sort_values(by=['date','variant']).reset_index(drop=True)

In [14]:
trend_df.head()

,date,variant,impression,user_id_x,session_id,event_time_x,page,event_id_x,user_id_y,click,...,event_type_x,order_id_x,amount,event_id_y,user_id,purchase,event_time,event_type_y,order_id_y,revenue
0,2025-07-01,0,57,57,57,57,57,9,9,9,...,9,0,9,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,2025-07-01,1,41,41,41,41,41,12,12,12,...,12,0,12,3.0,3.0,3.0,3.0,3.0,3.0,3.0
2,2025-07-02,0,137,137,137,137,137,24,24,24,...,24,0,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-07-02,1,88,88,88,88,88,21,21,21,...,21,0,21,6.0,6.0,6.0,6.0,6.0,6.0,6.0
4,2025-07-03,0,120,120,120,120,120,28,28,28,...,28,0,28,8.0,8.0,8.0,8.0,8.0,8.0,8.0


In [15]:
# TODO:
# click, purchase, revenue의 결측값을 0으로 채우세요.
trend_df[["click", "purchase", "revenue"]] = trend_df[["click", "purchase", "revenue"]].fillna(0)

In [16]:
# TODO:
# trend_df에서 CTR과 Conversion을 계산하세요.
trend_df["CTR"] = trend_df["click"] / trend_df["impression"]
trend_df["Conversion"] = trend_df["purchase"] / trend_df["impression"]

print(segment_df.head())
print(trend_df.head(10))
print(trend_df.groupby("variant")["date"].nunique())

   variant country   device  users  impression  click  purchase    revenue  \
0        0      DE  desktop     81         125   19.0       1.0   36077.91   
1        0      DE   mobile    128         212   44.0       9.0  261965.55   
2        0      DE   tablet     20          33    7.0       3.0   89288.56   
3        0      ES  desktop     36          64   12.0       1.0   16594.30   
4        0      ES   mobile     94         158   31.0       6.0  166486.76   

        CTR  Conversion  Revenue_per_user  
0  0.152000    0.008000        445.406296  
1  0.207547    0.042453       2046.605859  
2  0.212121    0.090909       4464.428000  
3  0.187500    0.015625        460.952778  
4  0.196203    0.037975       1771.135745  
         date  variant  impression  user_id_x  session_id  event_time_x  page  \
0  2025-07-01        0          57         57          57            57    57   
1  2025-07-01        1          41         41          41            41    41   
2  2025-07-02        0  

In [17]:
# TODO:
# segment_df와 trend_df를 각각 CSV 파일로 저장하세요.
segment_df.to_csv("../data/ab_segment_kpi.csv", index=False)
trend_df.to_csv("../data/ab_daily_trend.csv", index=False)